In [ ]:
import pandas as pd
import gpxpy
import os
import re

## Configuration
Remplacez par le chemin vers les fichiers à traiter

In [ ]:
data_path = './input/AEHO001-data.csv'
track_path = './input/Navionics_archive_export_23-09-25_au_26-09-25.gpx'
instrument_name = 'AE_HO_001' # Suivre le format du numéro de série (owner_type_numéro)
output_dir = "output"
csv_encoding = 'utf-8' # or 'latin-1' depending on your file encoding

## Étape 1 : Charger les données GPX

In [4]:
def load_gpx_tracks(file_path):
    with open(file_path, 'r') as gpx_file:
        gpx = gpxpy.parse(gpx_file)
    tracks = []
    for track in gpx.tracks:
        points = []
        for segment in track.segments:
            for point in segment.points:
                points.append({
                    'datetime': point.time,
                    'lat': point.latitude,
                    'long': point.longitude
                })
        if points:
            df_gpx = pd.DataFrame(points)
            df_gpx['datetime'] = pd.to_datetime(df_gpx['datetime']).dt.tz_convert('UTC')
            df_gpx = df_gpx.set_index('datetime')
            tracks.append(df_gpx)
    return tracks

gpx_tracks = load_gpx_tracks(track_path)

# Afficher la plage de temps de chaque tracée
for i, df_gpx in enumerate(gpx_tracks):
    print(f"Tracée {i+1} : de {df_gpx.index.min()} à {df_gpx.index.max()}")

Tracée 1 : de 2025-09-23 07:43:37.009000+00:00 à 2025-09-23 12:29:35.938000+00:00
Tracée 2 : de 2025-09-24 09:51:27.401000+00:00 à 2025-09-24 14:35:10.673000+00:00
Tracée 3 : de 2025-09-25 08:24:42.533000+00:00 à 2025-09-25 13:46:57.976000+00:00
Tracée 4 : de 2025-09-26 11:35:06.981000+00:00 à 2025-09-26 14:33:10.622000+00:00


## Étape 2 : Charger les données CSV

In [ ]:
def rename_columns_with_regex(df):
    # Mapping with regex patterns (the number is replaced by \\d+)
    col_map = {
        r'Température \(W-CTD-00 \d+\), °C': 'sea_temp',
        r'Pression absolue \(W-CTD-00 \d+\), kPa': 'air_pres',
        r'Conductivité électrique \(W-CTD-00 \d+\), µS/cm': 'ec_abs',
        r'Conductivité spécifique \(W-CTD-00 \d+\), µS/cm': 'ec25',
        r'Salinité \(W-CTD-00 \d+\), PSU': 'sea_sal',
        r'Matières solides dissoutes totales \(W-CTD-00 \d+\), mg/L': 'total_dissolved_solids',
    }
    new_columns = {}
    for col in df.columns:
        new_col = col
        for pattern, replacement in col_map.items():
            if re.fullmatch(pattern, col):
                new_col = replacement
                break
        new_columns[col] = new_col
    return df.rename(columns=new_columns)

df = pd.read_csv(data_path, low_memory=False, parse_dates=[1], date_format="%m/%d/%Y %H:%M:%S", delimiter=';', encoding=csv_encoding)

# Renommer les colonnes avec regex
# La colonne 'Date et heure (CEST)' reste renommée explicitement
if 'Date et heure (CEST)' in df.columns:
    df = df.rename(columns={'Date et heure (CEST)': 'datetime'})
df = rename_columns_with_regex(df)

# Supprimer les colonnes inutiles
df = df.drop(
    columns=['#', 'Hôte connecté', 'Fin du fichier', 'Démarré', 'Arrêté', 'Bouton Haut', 'Bouton Bas', 'Détection d’eau'],
    errors='ignore'
)

print(df.head())

# Convertir le fuseau horaire de CEST en UTC
df['datetime'] = df['datetime'].dt.tz_localize('Europe/Paris')
df['datetime'] = df['datetime'].dt.tz_convert('UTC')
df = df.set_index('datetime')

# Convertir la pression absolue de kPa en hPa
if 'air_pres' in df.columns:
    df['air_pres'] = df['air_pres'] * 10
    df['air_pres'] = df['air_pres'].round(2)


             datetime  sea_temp  air_pres   ec_abs     ec25  sea_sal  \
0 2025-07-21 18:44:42     11.32   102.380  35819.3  50262.0    31.53   
1 2025-07-21 18:45:42     11.29   102.379  35788.2  50254.4    31.52   
2 2025-07-21 18:46:42     11.27   102.338  35683.2  50138.5    31.44   
3 2025-07-21 18:47:42     11.25   102.296  35686.9  50179.7    31.46   
4 2025-07-21 18:48:42     11.22   102.336  35638.0  50147.0    31.43   

   total_dissolved_solids  
0                23282.52  
1                23262.33  
2                23194.10  
3                23196.49  
4                23164.71  


In [19]:
print("Plage de dates CSV :", df.index.min(), "à", df.index.max())

Plage de dates CSV : 2025-07-21 16:44:42+00:00 à 2025-09-26 15:37:28+00:00


## Étape 3 : Interpolation de la position des mesures à partir des traces GPS  

In [20]:
def interpolate_position(row, df_gpx):
    t = row.name
    idx = df_gpx.index.searchsorted(t, side='right') - 1
    if idx < 0 or idx >= len(df_gpx) - 1:
        return None, None  # Hors des limites
    point_A = df_gpx.iloc[idx]
    point_B = df_gpx.iloc[idx + 1]
    tA, tB = point_A.name, point_B.name
    latA, longA = point_A['lat'], point_A['long']
    latB, longB = point_B['lat'], point_B['long']
    ratio = (t - tA).total_seconds() / (tB - tA).total_seconds()
    lat = latA + (latB - latA) * ratio
    long = longA + (longB - longA) * ratio
    return lat, long

def interpolate_all_tracks(df, gpx_tracks):
    dfs = []
    for df_gpx in gpx_tracks:
        # Filtrer les mesures dans la plage de la tracée
        mask = (df.index >= df_gpx.index.min()) & (df.index <= df_gpx.index.max())
        df_sub = df[mask].copy()
        if df_sub.empty:
            continue
        df_sub[['lat', 'long']] = df_sub.apply(lambda row: interpolate_position(row, df_gpx), axis=1, result_type='expand')
        df_sub = df_sub.dropna(subset=['lat', 'long'])
        dfs.append(df_sub)
    if dfs:
        return pd.concat(dfs).sort_index()
    else:
        return pd.DataFrame()

In [21]:
df_interp = interpolate_all_tracks(df, gpx_tracks)

## Étape 4 : Exporter les données

In [22]:
os.makedirs(output_dir, exist_ok=True)

if not df_interp.empty:
    start_at = df_interp.index.min().strftime('%Y-%m-%d_%H:%M:%S')
    output_csv_final = os.path.join(output_dir, f"{instrument_name}_{start_at}_with_positions.csv")
    df_interp.to_csv(output_csv_final, index=False)
    print(f"Fichier exporté : {output_csv_final}")
else:
    print("Aucune donnée interpolée à exporter.")

Fichier exporté : output/AE_HO_001_2025-09-23_07:44:28_with_positions.csv
